In [78]:
import sys
sys.path.append("PyDI/PyDI")
from cleaners.rag_cleaner import RAGCleaner
import pandas as pd
import ollama
import os

In [79]:
# Knowledge base = datasets 1, 3, 4 combined
kb = pd.concat([
    pd.read_json("normalized_products/dataset_2_normalized.json"),
    pd.read_json("normalized_products/dataset_3_normalized.json"),
    pd.read_json("normalized_products/dataset_4_normalized.json")
], ignore_index=True)

# Dataset to clean
df = pd.read_json("normalized_products/dataset_1_normalized.json")

In [80]:
kb.head()

,id,brand,title,description,price,priceCurrency,cluster_id,url,title_description,model,...,manufacturing_process,memory_type,encryption_type,controller,storage_technology,color,form_factor,blocked_slots,case_material,delivery_scope
0,19126355,Gigabyte,Gigabyte NVIDIA GeForce RTX 3080 10GB GAMING O...,Gigabyte NVIDIA GeForce RTX 3080 GAMING OC 10G...,99999.99,GBP,1002037,https://www.scan.co.uk/products/gigabyte-nvidi...,Gigabyte NVIDIA GeForce RTX 3080 10GB GAMING O...,GeForce RTX 3080 10GB GAMING OC,...,None,GDDR6X,None,None,None,None,None,NaN,None,None
1,42841911,Western Digital,WD Blue 6TB Desktop Hard Disk Drive - 5400 RPM...,WD Blue 6TB Desktop Hard Disk Drive - 5400 RPM...,128.99,USD,1004942,https://www.newegg.com/blue-wd60ezaz-6tb/p/1Z4...,WD Blue 6TB Desktop Hard Disk Drive - 5400 RPM...,WD Blue,...,None,None,None,None,None,None,3.5 Inch,NaN,None,None
2,46775597,Corsair,CORSAIR - Force Series MP510 960GB M.2 SSD PCI...,CORSAIR Force Series MP510 960GB M.2 SSD PCIe ...,None,None,1007272,https://www.dindator.se/corsair-force-series-m...,CORSAIR - Force Series MP510 960GB M.2 SSD PCI...,Force Series MP510,...,None,None,None,None,None,None,M.2,NaN,None,None
3,33563069,None,"4TB Exos 7E8 ST4000NM0115, 7200 RPM, SATA 6Gb/...","4TB Exos 7E8 ST4000NM0115, 7200 RPM, SATA 6Gb/...",169.18,USD,1014052,https://www.avadirect.com/4TB-Exos-7E8-ST4000N...,"4TB Exos 7E8 ST4000NM0115, 7200 RPM, SATA 6Gb/...",Exos 7E8,...,None,None,None,None,None,None,3.5-Inch,NaN,None,None
4,56179024,Asus,ASUS GeForce GTX 1650 4GB Phoenix Boost Graphi...,"Available @ CCL, this ASUS GeForce GTX 1650 Ph...",1.4099E2,GBP,1014152,https://www.cclonline.com/product/279214/90YV0...,ASUS GeForce GTX 1650 4GB Phoenix Boost Graphi...,GeForce GTX 1650 4GB Phoenix Boost,...,None,None,None,None,None,None,None,NaN,None,None


In [81]:
class OllamaLLMWrapper:
    def __init__(self, model="llama3"):
        self.model = model

    def generate(self, prompt):
        response = ollama.chat(
            model=self.model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a structured data repair tool. "
                        "You ONLY output tokens in the format VALUE:<value>. "
                        "You NEVER write sentences, explanations, or any other text. "
                        "Your entire response must be a single line: VALUE:<value>"
                    )
                },
                {"role": "user", "content": prompt}
            ]
        )
        return response["message"]["content"].strip()

In [82]:
rag_cleaner = RAGCleaner(
    knowledge_base=kb,
    target_attributes=[
        "model_number",
        "storage_size",
        "bus_type",
        "interface_type",
        "form_factor",
        "vram",
        "storage_connection_type"
    ],
    llm=OllamaLLMWrapper("llama3"),
    top_k=2
)

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
Batches: 100%|██████████| 69/69 [00:02<00:00, 27.23it/s]


In [83]:
cols_to_track = [
    "model_number", "storage_size", "bus_type",
    "interface_type", "form_factor", "vram", "storage_connection_type"
]

In [84]:
null_before = df[cols_to_track].isnull().sum()
print("=== NULL COUNTS BEFORE CLEANING ===")
print(null_before)
print(f"Total: {null_before.sum()}\n")

=== NULL COUNTS BEFORE CLEANING ===
model_number               478
storage_size               241
bus_type                   184
interface_type             376
form_factor                395
vram                       583
storage_connection_type    399
dtype: int64
Total: 2656



In [85]:
df_test = df.head(2)

In [86]:
# cleaned_df = rag_cleaner.clean(df_test)
cleaned_df = rag_cleaner.clean(df)

Batches: 100%|██████████| 1/1 [00:00<00:00, 153.57it/s]
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"



--- CLEANING EVENT ---
Row: 0
Attribute: model_number
Old value: None
New value: VALUE:Gigabyte NVIDIA GeForce RTX 3080 Gaming OC 10GB Ampere Graphics Card


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.28it/s]
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"



--- CLEANING EVENT ---
Row: 0
Attribute: storage_size
Old value: nan
New value: VALUE:10.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.69it/s]
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


...


--- CLEANING EVENT ---
Row: 73
Attribute: interface_type
Old value: None
New value: VALUE:USB 3.0


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.56it/s]
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"



--- CLEANING EVENT ---
Row: 73
Attribute: vram
Old value: nan
New value: VALUE:nan


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.35it/s]


KeyboardInterrupt: 

In [ ]:
# print(cleaned_df[[
#     "title",
#     "model_number",
#     "storage_size",
#     "bus_type",
#     "interface_type",
#     "form_factor",
#     "vram",
#     "storage_connection_type"
# ]])

In [87]:
# After cleaning
null_after = cleaned_df[cols_to_track].isnull().sum()
print("\n=== NULL COUNTS AFTER CLEANING ===")
print(null_after)
print(f"Total: {null_after.sum()}\n")


=== NULL COUNTS AFTER CLEANING ===
model_number               0
storage_size               0
bus_type                   0
interface_type             0
form_factor                0
vram                       0
storage_connection_type    0
dtype: int64
Total: 0



In [88]:
# Summary
filled = null_before.sum() - null_after.sum()
total = null_before.sum()
print(f"=== SUMMARY ===")
print(f"Filled: {filled} / {total} ({100*filled/total:.1f}% fill rate)")

=== SUMMARY ===
Filled: 2656 / 2656 (100.0% fill rate)
